## 2026 EY AI & Data Challenge - TerraClimate Data Extraction (EXTENDED)

This notebook extracts **8 TerraClimate climate variables** that drive water chemistry.

| Variable | Description | Water Quality Relevance |
|---|---|---|
| `pet` | Potential Evapotranspiration | Atmospheric water demand |
| `ppt` | Precipitation (mm) | Dilution / concentration effects |
| `soil` | Soil Moisture (mm) | Mineral leaching |
| `tmax` | Maximum Temperature (°C) | Reaction rates |
| `tmin` | Minimum Temperature (°C) | Biological processes |
| `aet` | Actual Evapotranspiration | Real water consumed by vegetation |
| `def` | Climatic Water Deficit | Drought stress indicator |
| `srad` | Solar Radiation (W/m²) | Algae / photosynthesis driver |

**Outputs:** `terraclimate_features_training_EXTENDED.csv` and `terraclimate_features_validation_EXTENDED.csv`

> Each variable uses a **fresh dataset connection** to avoid API session timeouts.
> If a cell fails, simply re-run it — all other results are stored in memory.

## Load In Dependencies
Install required libraries. After the first run, restart the kernel before continuing.

### ⚠️ Action Required: Add Packages via Snowflake UI

Snowflake Notebooks do not support `!pip install`. Please add the following packages using the **Packages** dropdown at the top of this page:

**Required Packages:**
- `pystac`
- `pystac-client`
- `planetary-computer`
- `xarray`
- `zarr`
- `adlfs`
- `dask`
- `numcodecs`
- `scipy`
- `tqdm`
- `pandas`
- `numpy`
- `odc-stac` (for Landsat notebook)
- `rioxarray` (for Landsat notebook)

> [!IMPORTANT]
> After adding packages, you may need to **Restart Kernel** from the menu at the top.

## Setup: Imports, Session, Helper Functions

In [5]:
import snowflake
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.session import Session
import requests
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree
from tqdm import tqdm
import time
import os

try:
    session = get_active_session()
    print('✅ Linked to active Snowflake session')
except:
    try:
        session = Session.builder.getOrCreate()
        print('✅ Session created via builder')
    except:
        print('❌ No active session found. Ensure a Warehouse is selected.')

warnings.filterwarnings('ignore')
print('✅ Imports complete (using requests fallback for STAC)')

❌ No active session found. PLEASE SELECT A WAREHOUSE in the top-right dropdown of the Snowflake UI.
✅ Imports complete


In [ ]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def load_terraclimate_dataset():
    """Open TerraClimate Zarr dataset directly via requests and xarray."""
    # We fetch the zarr-abfs asset URL from the STAC collection endpoint
    collection_url = "https://planetarycomputer.microsoft.com/api/stac/v1/collections/terraclimate"
    resp = requests.get(collection_url).json()
    
    asset = resp['assets']['zarr-abfs']
    href = asset['href']
    storage_options = asset.get('xarray:storage_options', {})
    
    # For TerraClimate, we can often open without signing or it's public
    ds = xr.open_zarr(
        href, 
        storage_options=storage_options, 
        consolidated=True
    )
    return ds

def filterg(ds, var):
    """Filter to region & time, ensuring data exists."""
    # East Africa bounding box approximation
    # Lat: -5 to 5, Lon: 33 to 42
    subset = ds[var].sel(
        lat=slice(5.1, -5.1), 
        lon=slice(32.9, 42.1),
        time=slice('2011-01-01', '2015-12-31')
    )
    
    # Convert to dataframe for cKDTree mapping
    print(f"   Loading {var} data into memory (this can take a moment)...")
    df = subset.to_dataframe().reset_index()
    df = df.dropna(subset=[var])
    
    # Format time for easy matching
    df['time'] = pd.to_datetime(df['time']).dt.to_period('M').dt.to_timestamp()
    return df

def assign_nearest_climate(points_df, climate_df, var_name, quiet=False):
    """Vectorized nearest-neighbor mapping using cKDTree."""
    points = points_df.copy()
    points['Sample Date'] = pd.to_datetime(points['Sample Date']).dt.to_period('M').dt.to_timestamp()
    
    climate_values = []
    
    # Group by month to run spatial mapping once per month
    all_months = points['Sample Date'].unique()
    iterator = tqdm(all_months, desc=f'Mapping {var_name}') if not quiet else all_months
    
    for month in iterator:
        month_points = points[points['Sample Date'] == month]
        month_climate = climate_df[climate_df['time'] == month]
        
        if len(month_climate) == 0:
            climate_values.extend([np.nan] * len(month_points))
            continue
            
        # Build tree for this month's climate grid points
        tree = cKDTree(month_climate[['lat', 'lon']].values)
        _, idx = tree.query(month_points[['Latitude', 'Longitude']].values)
        
        climate_values.extend(month_climate.iloc[idx][var_name].values)
        
    return pd.DataFrame({var_name: climate_values})

print('✅ Helper functions loaded (Native requests + xarray)')

## Load Base Data

In [ ]:
Water_Quality_df = pd.read_csv('water_quality_training_dataset.csv')
Validation_df    = pd.read_csv('submission_template.csv')

print(f'Training samples : {Water_Quality_df.shape}')
print(f'Validation samples: {Validation_df.shape}')
display(Water_Quality_df.head(3))

# Dictionary to collect results
results_train = {}
results_val   = {}

## Extract Variables — One Cell Per Variable

Each cell opens a **fresh dataset connection** to avoid authentication timeouts.
Results accumulate in `results_train` and `results_val` dictionaries.
If a cell fails, simply re-run it.

⏱️ Estimated time per variable: **25–35 minutes** (training 9,319 pts + validation 200 pts)

In [ ]:
# ── VARIABLE 1: PET — Potential Evapotranspiration ───────────────────────────
print('═' * 70)
print('  EXTRACTING: pet — Potential Evapotranspiration')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'pet')

pet_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'pet')
pet_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'pet', quiet=True)

results_train['pet'] = pet_train['pet']
results_val['pet']   = pet_val['pet']

print(f'\n✅ PET done  |  train NaN: {pet_train["pet"].isna().mean():.1%}  |  val NaN: {pet_val["pet"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 2: PPT — Precipitation ─────────────────────────────────────────
print('═' * 70)
print('  EXTRACTING: ppt — Precipitation (mm)')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'ppt')

ppt_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'ppt')
ppt_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'ppt', quiet=True)

results_train['ppt'] = ppt_train['ppt']
results_val['ppt']   = ppt_val['ppt']

print(f'\n✅ PPT done  |  train NaN: {ppt_train["ppt"].isna().mean():.1%}  |  val NaN: {ppt_val["ppt"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 3: SOIL — Soil Moisture ─────────────────────────────────────────
print('═' * 70)
print('  EXTRACTING: soil — Soil Moisture (mm)')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'soil')

soil_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'soil')
soil_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'soil', quiet=True)

results_train['soil'] = soil_train['soil']
results_val['soil']   = soil_val['soil']

print(f'\n✅ SOIL done  |  train NaN: {soil_train["soil"].isna().mean():.1%}  |  val NaN: {soil_val["soil"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 4: TMAX — Maximum Temperature ───────────────────────────────────
# TerraClimate stores temperature scaled x10; divide by 10 to get °C
print('═' * 70)
print('  EXTRACTING: tmax — Maximum Temperature (stored x10)')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'tmax')

tmax_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'tmax')
tmax_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'tmax', quiet=True)

tmax_train['tmax'] = tmax_train['tmax'] / 10.0
tmax_val['tmax']   = tmax_val['tmax']   / 10.0

results_train['tmax'] = tmax_train['tmax']
results_val['tmax']   = tmax_val['tmax']

print(f'\n✅ TMAX done  |  train NaN: {tmax_train["tmax"].isna().mean():.1%}  |  val NaN: {tmax_val["tmax"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 5: TMIN — Minimum Temperature ───────────────────────────────────
print('═' * 70)
print('  EXTRACTING: tmin — Minimum Temperature (stored x10)')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'tmin')

tmin_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'tmin')
tmin_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'tmin', quiet=True)

tmin_train['tmin'] = tmin_train['tmin'] / 10.0
tmin_val['tmin']   = tmin_val['tmin']   / 10.0

results_train['tmin'] = tmin_train['tmin']
results_val['tmin']   = tmin_val['tmin']

print(f'\n✅ TMIN done  |  train NaN: {tmin_train["tmin"].isna().mean():.1%}  |  val NaN: {tmin_val["tmin"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 6: AET — Actual Evapotranspiration (NEW) ────────────────────────
# aet = how much water vegetation actually evaporated (constrained by soil moisture).
# Contrast with pet (demand): aet close to pet => well-watered; aet << pet => drought.
# Linked to nutrient cycling and biological oxygen demand in rivers.
print('═' * 70)
print('  EXTRACTING: aet — Actual Evapotranspiration (mm)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'aet')

aet_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'aet')
aet_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'aet', quiet=True)

results_train['aet'] = aet_train['aet']
results_val['aet']   = aet_val['aet']

print(f'\n✅ AET done  |  train NaN: {aet_train["aet"].isna().mean():.1%}  |  val NaN: {aet_val["aet"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 7: DEF — Climatic Water Deficit (NEW) ───────────────────────────
# def = pet - aet: the unmet atmospheric demand for water.
# High def => drought => concentrates dissolved minerals => affects EC and TA targets.
print('═' * 70)
print('  EXTRACTING: def — Climatic Water Deficit (mm)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'def')

def_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'def')
def_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'def', quiet=True)

results_train['def'] = def_train['def']
results_val['def']   = def_val['def']

print(f'\n✅ DEF done  |  train NaN: {def_train["def"].isna().mean():.1%}  |  val NaN: {def_val["def"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 8: SRAD — Solar Radiation (NEW) ─────────────────────────────────
# srad = downward surface solar radiation (W/m2).
# Drives algal growth and water temperature, influencing DRP concentrations.
print('═' * 70)
print('  EXTRACTING: srad — Solar Radiation (W/m2)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'srad')

srad_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'srad')
srad_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'srad', quiet=True)

results_train['srad'] = srad_train['srad']
results_val['srad']   = srad_val['srad']

print(f'\n✅ SRAD done  |  train NaN: {srad_train["srad"].isna().mean():.1%}  |  val NaN: {srad_val["srad"].isna().mean():.1%}')

## Combine & Save
Run this cell **only after all 8 extraction cells have completed successfully**.

In [ ]:
# ── VARIABLE 9: Q — Runoff (mm) (NEW) ────────────────────────
print('═' * 70)
print('  EXTRACTING: q — Runoff (mm)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'q')

q_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'q')
q_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'q', quiet=True)

results_train['q'] = q_train['q']
results_val['q']   = q_val['q']

print(f'\n✅ Q done  |  train NaN: {q_train["q"].isna().mean():.1%}  |  val NaN: {q_val["q"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 10: SWE — Snow Water Equivalent (mm) (NEW) ────────────────────────
print('═' * 70)
print('  EXTRACTING: swe — Snow Water Equivalent (mm)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'swe')

swe_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'swe')
swe_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'swe', quiet=True)

results_train['swe'] = swe_train['swe']
results_val['swe']   = swe_val['swe']

print(f'\n✅ SWE done  |  train NaN: {swe_train["swe"].isna().mean():.1%}  |  val NaN: {swe_val["swe"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 11: VAP — Vapor Pressure (kPa) (NEW) ────────────────────────
print('═' * 70)
print('  EXTRACTING: vap — Vapor Pressure (kPa)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'vap')

vap_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'vap')
vap_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'vap', quiet=True)

results_train['vap'] = vap_train['vap']
results_val['vap']   = vap_val['vap']

print(f'\n✅ VAP done  |  train NaN: {vap_train["vap"].isna().mean():.1%}  |  val NaN: {vap_val["vap"].isna().mean():.1%}')

In [ ]:
# ── VARIABLE 12: WS — Wind Speed (m/s) (NEW) ────────────────────────
print('═' * 70)
print('  EXTRACTING: ws — Wind Speed (m/s)  [NEW]')
print('═' * 70)

ds = load_terraclimate_dataset()
tc_grid = filterg(ds, 'ws')

ws_train = assign_nearest_climate(Water_Quality_df, tc_grid.copy(), 'ws')
ws_val   = assign_nearest_climate(Validation_df,    tc_grid.copy(), 'ws', quiet=True)

results_train['ws'] = ws_train['ws']
results_val['ws']   = ws_val['ws']

print(f'\n✅ WS done  |  train NaN: {ws_train["ws"].isna().mean():.1%}  |  val NaN: {ws_val["ws"].isna().mean():.1%}')

In [ ]:
vars_extracted = list(results_train.keys())
print(f'Variables extracted: {vars_extracted}')

train_climate = pd.DataFrame(results_train)
train_climate.insert(0, 'Latitude',    Water_Quality_df['Latitude'].values)
train_climate.insert(1, 'Longitude',   Water_Quality_df['Longitude'].values)
train_climate.insert(2, 'Sample Date', Water_Quality_df['Sample Date'].values)

val_climate = pd.DataFrame(results_val)
val_climate.insert(0, 'Latitude',    Validation_df['Latitude'].values)
val_climate.insert(1, 'Longitude',   Validation_df['Longitude'].values)
val_climate.insert(2, 'Sample Date', Validation_df['Sample Date'].values)

print(f'Training shape  : {train_climate.shape}')
print(f'Validation shape: {val_climate.shape}')

nans_train = train_climate[vars_extracted].isna().mean().round(3) * 100
print('\nTraining NaN % per variable:')
print(nans_train.to_string())

display(train_climate.head())

In [ ]:
train_climate.to_csv('terraclimate_features_training_EXTENDED.csv', index=False)
val_climate.to_csv('terraclimate_features_validation_EXTENDED.csv', index=False)

print('✅ Saved:')
print('   📁 terraclimate_features_training_EXTENDED.csv')
print('   📁 terraclimate_features_validation_EXTENDED.csv')

In [ ]:
# ── PERSISTENCE (TRAINING): Write to Snowflake Table ─────────────────────────
table_name = 'TERRACLIMATE_EXTENDED_TRAINING'
session.write_pandas(train_climate, table_name, auto_create_table=True, overwrite=True)
print(f'✅ Training data persisted to table: {table_name}')
print('👉 To get CSV: Run the export cell below.')

In [ ]:
# ── PERSISTENCE (VALIDATION): Write to Snowflake Table ───────────────────────
table_name = 'TERRACLIMATE_EXTENDED_VALIDATION'
session.write_pandas(val_climate, table_name, auto_create_table=True, overwrite=True)
print(f'✅ Validation data persisted to table: {table_name}')

## Verification

In [ ]:
expected_vars = ['pet', 'ppt', 'soil', 'tmax', 'tmin', 'aet', 'def', 'srad']

df_train_check = pd.read_csv('terraclimate_features_training_EXTENDED.csv')
df_val_check   = pd.read_csv('terraclimate_features_validation_EXTENDED.csv')

print('═' * 60)
print('TRAINING FILE VERIFICATION')
print(f'  Shape         : {df_train_check.shape}  (expected 9319 rows)')
missing = [v for v in expected_vars if v not in df_train_check.columns]
print(f'  Missing vars  : {missing if missing else "None ✅"}')
nans = df_train_check[expected_vars].isna().mean().round(3) * 100
print('  NaN % per var :')
print(nans.to_string())

print('\nVALIDATION FILE VERIFICATION')
print(f'  Shape         : {df_val_check.shape}  (expected 200 rows)')
missing_v = [v for v in expected_vars if v not in df_val_check.columns]
print(f'  Missing vars  : {missing_v if missing_v else "None ✅"}')

print('\n✅ Verification complete — download files and place in Jupyter Notebook Package/ for modeling.')

In [ ]:
# ── EXPORT TO CSV FOR LOCAL DOWNLOAD ─────────────────────────────────────────
train_climate.to_csv('terraclimate_features_training_EXTENDED.csv', index=False)
val_climate.to_csv('terraclimate_features_validation_EXTENDED.csv', index=False)
print('✅ CSV files generated in local sidebar.')